<a href="https://colab.research.google.com/github/jc13605-0721/ECE_9533_LLM4ChipDesign/blob/main/HW2_ROME/rome_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Initial Setup

In [ ]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 4.6 MB/s eta 0:00:00
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,793 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:13 http://a

In [ ]:
#@title Select Model
#define the model to be used
model_choice = "gpt-4o"
#model_choice = "claude-3-7-sonnet-2025029"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"

In [ ]:
#@title Utility functions

import sys
import os
import re
import time

import openai
import anthropic
import google.genai.errors
from google import genai
from google.genai import types
from abc import ABC, abstractmethod


################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""
    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""
    def __init__(self, model_id=None):
        super().__init__()
        # model_choice is defined in your "Select Model" cell
        self.model_id = model_id if model_id is not None else model_choice
        openai.api_key = os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()

    def generate(self, conversation: Conversation, num_choices=1):
        # NOTE: Your original code converts everything to role="user".
        # Keeping this behavior for compatibility.
        messages = [{"role": "user", "content": msg["content"]} for msg in conversation.get_messages()]
        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )
        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.model_id = model_id if model_id is not None else model_choice
        self.client = anthropic.Anthropic(api_key=os.environ['CLAUDE_API_KEY'])

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20
        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=[{"role": msg["role"], "content": msg["content"]} for msg in conversation.get_messages()]
                ).content[0].text
                return output
            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)
        print(f"Failed, exceeded max retries {max_retries}")
        return ""


class Gemini(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.model_id = model_id if model_id is not None else model_choice
        self.gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text
        return output


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################

def find_verilog_modules(text):
    """
    Robustly match Verilog modules, including:
      - module name(...); ... endmodule
      - module name; ... endmodule          (common in testbenches)
      - module name #(...)(...); ... endmodule
    """
    pattern = r'\bmodule\b\s+\w+\s*(?:#\s*\([^;]*\)\s*)?(?:\([^;]*\)\s*)?;.*?\bendmodule\b'
    return re.findall(pattern, text, re.DOTALL)


def write_code_blocks_to_file(markdown_string, module_name, filename):
    """
    Robust file writer:
      1) Prefer fenced blocks ```verilog ... ```
      2) Fallback to regex module...endmodule extraction
      3) Last resort: write raw response (no exit), so debugging is possible
    """
    # 1) Try fenced code blocks first
    code_blocks = re.findall(r"```(?:verilog)?\s*([\s\S]*?)```", markdown_string, flags=re.IGNORECASE)
    extracted = None

    if code_blocks:
        extracted_blocks = [b.strip() for b in code_blocks if b.strip()]
        if extracted_blocks:
            extracted = "\n\n".join(extracted_blocks)

    # 2) Fallback: module...endmodule blocks
    if not extracted:
        modules = find_verilog_modules(markdown_string)
        if modules:
            extracted = "\n\n".join([m.strip() for m in modules])

    # 3) Last resort: raw text
    if not extracted:
        print("No Verilog code found; writing raw response for debugging (no exit).")
        extracted = markdown_string.strip()

    # Ensure output directory exists
    outdir = os.path.dirname(filename)
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    with open(filename, 'w') as f:
        f.write(extracted + "\n")


def generate_verilog(conv, model_type, model_id=""):
    if model_type == "ChatGPT":
        model = ChatGPT(model_id=model_id if model_id else None)
    elif model_type == "Claude":
        model = Claude(model_id=model_id if model_id else None)
    elif model_type == "Gemini":
        model = Gemini(model_id=model_id if model_id else None)
    else:
        raise ValueError("Invalid model type")
    return model.generate(conv)

In [ ]:
import subprocess
import sys
import os
import time
import numpy as np

def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, outdir="", log=None, prev_module=None):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)

    # ---- System prompt: FORCE fenced code block ----
    sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, output ONLY ONE Verilog module. "
        "CRITICAL FORMATTING: Return ONLY a single fenced code block in the form:\n"
        "```verilog\n<full module code>\n```\n"
        "Do NOT output any text outside the code block."
    )

    if model_type == "ChatGPT":
        conv.add_message("system", sys_prompt)
    elif model_type == "Claude":
        conv.add_message("user", sys_prompt)

    # ---- User prompt (also reinforce formatting) ----
    conv.add_message("user", design_prompt + "\n\nReturn ONLY a ```verilog``` fenced code block.")

    success = False
    timeout = False
    iterations = 0
    timelist_total, timelist_gen, timelist_error = [], [], []

    filename = os.path.join(outdir, module + ".v")
    status = ""

    while not (success or timeout):
        start_total = time.time()
        response = generate_verilog(conv, model_type)
        end_gen = time.time()
        start_error = time.time()

        # Prepend previous module text if needed (your original behavior)
        if prev_module is None:
            conv.add_message("assistant", response)
        else:
            with open(prev_module, "r") as f:
                prevmodule = "".join(f.read())
            response = prevmodule + "\n" + response
            conv.add_message("assistant", response)

        # Write module file from fenced code
        write_code_blocks_to_file(response, module, filename)

        # ---- Generate TB (ALLOW TB + FORCE fenced block) ----
        tb_prompt = (
            f"Generate a self-checking Verilog testbench for module {module}. "
            f"The DUT is defined in {module}.v and will be compiled together. "
            "CRITICAL:\n"
            "1) Return ONLY ONE fenced code block:\n"
            "```verilog\n<testbench code>\n```\n"
            "2) No text outside the code block.\n"
            "3) The testbench must end by printing exactly the token passed! on its own line (e.g., $display(\"passed!\");)\n"
            "4) The testbench must call $finish.\n"
        )

        conv_tb = Conversation()
        # Important: TB system prompt MUST NOT forbid TB
        if model_type == "ChatGPT":
            conv_tb.add_message("system", "You generate Verilog TESTBENCH code only. Use a single ```verilog``` fenced code block and no extra text.")
        else:
            conv_tb.add_message("user", "You generate Verilog TESTBENCH code only. Use a single ```verilog``` fenced code block and no extra text.")

        conv_tb.add_message("user", tb_prompt)

        tb_response = generate_verilog(conv_tb, model_type)
        write_code_blocks_to_file(tb_response, module, testbench)

        # Compile
        proc = subprocess.run(
            ["iverilog", "-g2012", "-o", os.path.join(outdir, module), filename, testbench],
            capture_output=True, text=True
        )

        success = False
        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)
            message = "The testbench failed to compile. Please fix the module. The output of iverilog is as follows:\n" + proc.stderr

        elif proc.stderr.strip() != "":
            status = "Warnings compiling testbench"
            print(status)
            message = "The testbench compiled with warnings. Please fix the module. The output of iverilog is as follows:\n" + proc.stderr

        else:
            # Run
            runp = subprocess.run(["vvp", os.path.join(outdir, module)], capture_output=True, text=True)
            out = (runp.stdout or "").strip()

            # ---- Robust pass/fail check (no [-2]) ----
            if "passed!" not in out:
                status = "Error running testbench"
                print(status)
                message = "The testbench simulated, but did not print passed!. Please fix the module. Simulation output is:\n" + out
            else:
                status = "Testbench ran successfully"
                print(status)
                message = ""
                success = True

        # Write log
        with open(os.path.join(outdir, "log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write('\n'.join(str(i) for i in conv.get_messages()))
            file.write('\n\n Iteration status: ' + status + '\n')

        if not success:
            if iterations > 0:
                # remove previous user error message and assistant response (keep consistent with your logic)
                conv.remove_message(2)
                conv.remove_message(2)
            conv.add_message("user", message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1
        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))
    return (np.sum(timelist_total), np.sum(timelist_gen), np.sum(timelist_error))

In [ ]:
#@title Hierarchical Loop
def hier_gen(submods,max_iterations=10):
  totaltime = []
  gentime = []
  errortime = []
  done =""
  for i in range(len(submods)):
    curr = submods[i][1]
    fcurr = submods[i][0]
    iocurr = submods[i][2]
    overall = submods[-1][1]
    if not os.path.isdir(fcurr):
      os.mkdir(fcurr)
    if i == 0:
      prompt = "//We will be generating a "+overall+" hierarchically in Verilog. Please begin by generating a "+curr+" defined as follows:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    elif i != len(submods)-1:
      fprev = submods[i-1][0]
      filecurr = "./"+fprev+"/"+fprev+".v"
      with open(filecurr,"r") as f:
        modulef = "".join(f.read())
      prompt = "//We are generating a "+overall+" hierarchically in Verilog. We have generated "+done+" defined as follows:"
      prompt = prompt + modulef
      prompt = prompt +"\n//Please include the previous module(s) in your response and use them to hierarchically generate a "+curr+" defined as:\nmodule "+fcurr+"("+iocurr+")\n//Insert code here\nendmodule"
    module = fcurr
    testbench = "./"+fcurr+"tb.v"
    model = os.environ["MODEL"]
    outdir = "./"+fcurr
    log = "./"+fcurr+"/log.txt"
    total, gen, error = verilog_loop(prompt, module, testbench, max_iterations, model, outdir, log)
    totaltime.append(total)
    gentime.append(gen)
    errortime.append(error)
    done = done + curr+", "
  print("Overall Total time: ",np.sum(totaltime))
  print("Overall Generation Time: ",np.sum(gentime))
  print("Overall Error handling time: ",np.sum(errortime))

# Setting the API Key

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ["OPENAI_API_KEY"], "OPENAI_API_KEY is empty"

os.environ["MODEL"] = "ChatGPT"


#Mux Hierarchy Example

In [ ]:
#@title Submodules


### Each step is structured as ["filename","natural language description"]
submodules = [
    ["mux2to1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4to1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8to1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
]

In [ ]:
hier_gen(submodules)

Error compiling testbench
Error running testbench
Error compiling testbench
Testbench ran successfully
Total time:  14.73368787765503
Generation time:  3.2390635013580322
Error handling time:  11.494621992111206
Testbench ran successfully
Total time:  4.08673620223999
Generation time:  1.3256149291992188
Error handling time:  2.761120557785034
Error compiling testbench
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  19.131274700164795
Generation time:  7.397238731384277
Error handling time:  11.734032392501831
Overall Total time:  37.951698780059814
Overall Generation Time:  11.961917161941528
Overall Error handling time:  25.98977494239807


In [ ]:
!vvp mux2to1/mux2to1
!vvp mux4to1/mux4to1
!vvp mux8to1/mux8to1

passed!
passed!
passed!


In [ ]:
submodules_part2 = [
    ["half_adder", "half adder", "input wire a, input wire b, output wire sum, output wire carry"],
    ["full_adder", "full adder", "input wire a, input wire b, input wire cin, output wire sum, output wire cout"],
    ["adder4", "4-bit ripple-carry adder", "input wire [3:0] a, input wire [3:0] b, input wire cin, output wire [3:0] sum, output wire cout"],
    ["adder8", "8-bit ripple-carry adder", "input wire [7:0] a, input wire [7:0] b, input wire cin, output wire [7:0] sum, output wire cout"],
]

#Part II

In [ ]:
hier_gen(submodules_part2, max_iterations=15)

Error running testbench
Testbench ran successfully
Total time:  6.113650798797607
Generation time:  2.077577829360962
Error handling time:  4.036072492599487
Testbench ran successfully
Total time:  4.866234302520752
Generation time:  1.356428861618042
Error handling time:  3.5098049640655518
Warnings compiling testbench
Testbench ran successfully
Total time:  11.10983395576477
Generation time:  4.713640213012695
Error handling time:  6.396192789077759
Error compiling testbench
Warnings compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Error compiling testbench
Testbench ran successfully
Total time:  63.65125012397766
Generation time:  32.85202431678772
Error handling time:  30.799220085144043
Overall Total time:  85.74096918106079
Overall Generation Time:  40.99967122077942
Overall Error handling time:  44.74129033088684


In [ ]:
!vvp half_adder/half_adder
!vvp full_adder/full_adder
!vvp adder4/adder4
!vvp adder8/adder8

passed!
passed!
passed!
passed!


In [ ]:
!grep -n "output of iverilog" -n adder8/log_iter_2.txt -A 80

4:{'role': 'user', 'content': 'The testbench compiled with warnings. Please fix the module. The output of iverilog is as follows:\n./adder8tb.v:11: warning: Port 4 (sum) of adder8 expects 8 bits, got 9.\n./adder8tb.v:11:        : Padding 1 high bits of the expression.\n'}
5-{'role': 'assistant', 'content': '```verilog\nmodule half_adder(input wire a, input wire b, output wire sum, output wire carry);\n    // Logic for sum and carry\n    assign sum = a ^ b;    // XOR for sum\n    assign carry = a & b;  // AND for carry\nendmodule\n\nmodule full_adder(input wire a, input wire b, input wire cin, output wire sum, output wire cout);\n    wire sum1, carry1, carry2;\n\n    // Instantiate two half adders\n    half_adder ha1 (.a(a), .b(b), .sum(sum1), .carry(carry1));\n    half_adder ha2 (.a(sum1), .b(cin), .sum(sum), .carry(carry2));\n\n    // Calculate the final carry out\n    assign cout = carry1 | carry2;\nendmodule\n\nmodule adder8(input wire [7:0] a, input wire [7:0] b, input wire cin, ou

In [ ]:
!iverilog -g2012 -o sim8 adder8/adder8.v adder8tb.v